# Статистический анализ всех признаков

Ноутбук читает итоговый датасет модели 1 и строит один общий паспорт. Никакие колонки заранее не удаляются.

В итоговом CSV есть два типа строк:

- `dataset_metric` — общие показатели датасета;
- `feature` — статистика одной колонки.

Для каждого признака считаются заполненность, количество уникальных значений, основные статистики, частые категории и предварительная связь с `insured_sum`. ID, служебные колонки и возможная утечка не удаляются, а получают отдельную роль и комментарий.

Реальные адреса, ИНН, названия компаний, номера договоров и кадастровые номера в `top_values` не выводятся.

Под «всеми признаками» здесь понимаются все колонки, которые уже есть во входном CSV. Ноутбук не запрашивает дополнительные колонки из Сферы или КХД.


In [ ]:
%pip install pandas numpy openpyxl

In [ ]:
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)


# 1. Файлы и настройки


In [ ]:
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

results_dir = project_root / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
input_file = results_dir / 'датасет_строгий_адрес_CDI_ЕГРН.csv'
output_file = results_dir / 'паспорт_всех_признаков.csv'
dictionary_file = (
    project_root
    / 'МАТЕРИАЛЫ_ПРОЕКТА'
    / '00_входные_материалы'
    / 'колонки_таблиц.xlsx'
)

target_column = 'insured_sum'
csv_separator = ';'
csv_encoding = 'utf-8-sig'
top_values_limit = 10

print('Входной файл:', input_file)
print('Итоговый паспорт:', output_file)


Если файл датасета называется иначе, измени только значение `input_file` в предыдущей ячейке.


In [ ]:
if not input_file.exists():
    available_files = sorted(results_dir.glob('*.csv'))
    available_names = [path.name for path in available_files]
    raise FileNotFoundError(
        'Не найден входной CSV. Укажи правильный путь в input_file. '
        f'CSV в папке результатов: {available_names}'
    )

df = pd.read_csv(
    input_file,
    sep=csv_separator,
    encoding=csv_encoding,
    low_memory=False,
)
df.columns = [str(column).strip() for column in df.columns]

if target_column not in df.columns:
    raise ValueError(
        f'В датасете нет целевой колонки {target_column}'
    )

print('Строк:', len(df))
print('Колонок:', len(df.columns))


# 2. Словарь источников


In [ ]:
# словарь нужен только для понятных названий и исходных таблиц
dictionary_rows = pd.DataFrame()

if dictionary_file.exists():
    dictionary_parts = []
    for sheet_name in ['Сфера', 'КХД 1.0']:
        part = pd.read_excel(dictionary_file, sheet_name=sheet_name)
        part['SOURCE_SYSTEM'] = sheet_name
        dictionary_parts.append(part)
    dictionary_rows = pd.concat(dictionary_parts, ignore_index=True)
    dictionary_rows.columns = [
        str(column).strip().upper()
        for column in dictionary_rows.columns
    ]
    dictionary_rows['TABLE_NAME_KEY'] = (
        dictionary_rows['TABLE_NAME'].astype('string').str.upper()
    )
    dictionary_rows['COLUMN_NAME_KEY'] = (
        dictionary_rows['COLUMN_NAME'].astype('string').str.lower()
    )

print('Строк в словаре:', len(dictionary_rows))


In [ ]:
technical_columns = {
    'sphere_row_id',
    'row_source',
    'contract_link_status',
    'contract_count',
    'has_contract',
    'has_address',
    'has_target',
    'target_status',
    'source_address',
    'input_address',
    'cdi_house_fias_candidate_count',
    'cdi_match_status',
    'cdi_is_unique_match',
    'egrn_address_candidate_count',
    'egrn_candidate_count',
    'egrn_is_unique_match',
    'egrn_match_method',
    'pipeline_match_status',
    'external_snapshot_at',
}

identifier_columns = {
    'contract_id',
    'contract_number',
    'previous_contract_id',
    'root_contract_id',
    'request_id',
    'task_id',
    'task_object_link_id',
    'characteristics_id',
    'object_id',
    'geo_address_id',
    'policyholder_id',
    'corporate_crm_id',
    'policyholder_inn',
    'policyholder_cdi_id',
    'policyholder_ogrn',
    'policyholder_kpp',
    'address_dgis_id',
    'cdi_fias_id_house',
    'egrn_cad_ind',
    'egrn_cadaster',
    'egrn_fias_id_house',
}

target_derived_columns = {
    'has_target',
    'target_status',
    'condition_min_insured_sum',
    'condition_max_insured_sum',
    'task_object_insured_sum',
    'contract_insured_sum',
}

sensitive_pattern = re.compile(
    r'(address|адрес|inn|ogrn|kpp|cadaster|contract_number|'
    r'policyholder_name|company_name|description|comment|json|'
    r'phone|email|(^|_)id($|_))',
    flags=re.IGNORECASE,
)

date_pattern = re.compile(
    r'(date|time|timestamp|d_create|d_change|d_delete|_start|_end|as_of)',
    flags=re.IGNORECASE,
)


def infer_source(column):
    if column in technical_columns:
        return 'Технический'
    if column.startswith('egrn_'):
        return 'ЕГРН'
    if column.startswith('cdi_'):
        return 'CDI'
    return 'Сфера'


def infer_role(column):
    if column == target_column:
        return 'target'
    if column == 'as_of_date':
        return 'split_key'
    if column in technical_columns:
        return 'service'
    if column in identifier_columns or column.endswith('_id'):
        return 'identifier'
    return 'feature'


In [ ]:
exact_table_map = {
    'insured_sum': 'BASE_INSURANCE_OBJECT_CONDITIONS',
    'full_address': 'BASE_GEO_ADDRESS',
    'original_address': 'BASE_INSURANCE_OBJECT',
    'object_description': 'BASE_INSURANCE_OBJECT',
    'total_area': 'BASE_INSURANCE_OBJECT_CHARACTERISTICS',
    'policyholder_inn': 'BPS_CONTRACTOR',
    'crm_industry': 'BPS_CORPORATE_CRM',
}


def dictionary_info(column, source):
    if dictionary_rows.empty or source in {'CDI', 'Технический'}:
        return None, None, None

    lookup_column = column
    table_filter = None

    if source == 'ЕГРН':
        lookup_column = column.removeprefix('egrn_')
        table_filter = 'EGRN_DATA'
    elif column in exact_table_map:
        table_filter = exact_table_map[column]

    matches = dictionary_rows.loc[
        dictionary_rows['COLUMN_NAME_KEY'].eq(lookup_column.lower())
    ].copy()

    if table_filter is not None:
        matches = matches.loc[
            matches['TABLE_NAME_KEY'].eq(table_filter)
        ]

    if matches.empty:
        return None, None, None

    row = matches.iloc[0]
    return (
        row.get('TABLE_NAME'),
        row.get('COLUMN_NAME'),
        row.get('COLUMN_COMMENTS'),
    )


# 3. Функции статистического анализа


In [ ]:
def clean_string_series(series):
    return (
        series.astype('string')
        .str.replace('\u00a0', ' ', regex=False)
        .str.strip()
        .replace('', pd.NA)
    )


def numeric_view(series):
    if pd.api.types.is_bool_dtype(series):
        return pd.Series(np.nan, index=series.index), 0.0
    if pd.api.types.is_numeric_dtype(series):
        converted = pd.to_numeric(series, errors='coerce')
    else:
        cleaned = (
            clean_string_series(series)
            .str.replace(' ', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        converted = pd.to_numeric(cleaned, errors='coerce')

    original_filled = clean_string_series(series).notna().sum()
    ratio = (
        converted.notna().sum() / original_filled
        if original_filled
        else 0.0
    )
    return converted, ratio


def datetime_view(series):
    converted = pd.to_datetime(series, errors='coerce')
    original_filled = clean_string_series(series).notna().sum()
    ratio = (
        converted.notna().sum() / original_filled
        if original_filled
        else 0.0
    )
    return converted, ratio


def infer_data_type(column, series, role):
    filled = series.dropna()
    if filled.empty:
        return 'unknown'
    if 'json' in column.lower():
        return 'json'
    if pd.api.types.is_bool_dtype(series):
        return 'boolean'

    text_values = set(
        clean_string_series(filled)
        .dropna()
        .str.lower()
        .unique()
        .tolist()
    )
    boolean_values = {
        'true', 'false', 't', 'f', 'yes', 'no', 'да', 'нет', '0', '1'
    }
    if text_values and text_values.issubset(boolean_values):
        return 'boolean'

    if date_pattern.search(column):
        _, date_ratio = datetime_view(series)
        if date_ratio >= 0.70:
            return 'datetime'

    if role != 'identifier':
        _, numeric_ratio = numeric_view(series)
        if numeric_ratio >= 0.95:
            return 'numeric'

    unique_count = filled.nunique(dropna=True)
    if unique_count <= 100 or unique_count / len(filled) <= 0.20:
        return 'category'
    return 'text'


In [ ]:
def source_available_mask(frame, source):
    if source == 'ЕГРН' and 'egrn_is_unique_match' in frame.columns:
        return pd.to_numeric(
            frame['egrn_is_unique_match'],
            errors='coerce',
        ).eq(1)
    if source == 'CDI' and 'cdi_is_unique_match' in frame.columns:
        return pd.to_numeric(
            frame['cdi_is_unique_match'],
            errors='coerce',
        ).eq(1)
    return pd.Series(True, index=frame.index)


def safe_top_values(series, feature, role, limit=10):
    if role == 'identifier' or sensitive_pattern.search(feature):
        return 'скрыто: конфиденциальное поле'

    clean = clean_string_series(series).fillna('NULL')
    counts = clean.value_counts(dropna=False).head(limit)
    total = len(clean)
    parts = []

    for value, count in counts.items():
        value_text = str(value).replace('\n', ' ').replace('|', '/')[:80]
        percent = count / total * 100 if total else 0
        parts.append(f'{value_text} [{count}, {percent:.2f}%]')

    return ' | '.join(parts)


def eta_squared(categories, target):
    pair = pd.DataFrame({'category': categories, 'target': target}).dropna()
    if len(pair) < 10 or pair['category'].nunique() < 2:
        return np.nan, len(pair)

    overall_mean = pair['target'].mean()
    total_variation = ((pair['target'] - overall_mean) ** 2).sum()
    if total_variation == 0:
        return np.nan, len(pair)

    grouped = pair.groupby('category')['target'].agg(['count', 'mean'])
    between_variation = (
        grouped['count'] * (grouped['mean'] - overall_mean) ** 2
    ).sum()
    return float(between_variation / total_variation), len(pair)


In [ ]:
def leakage_assessment(column, source, role):
    if role == 'target':
        return 'not_applicable', 'целевая переменная'
    if column in target_derived_columns:
        return 'high', 'поле напрямую связано с расчётом или наличием target'
    if role == 'identifier':
        return 'not_applicable', 'технический идентификатор'
    if role == 'service':
        return 'medium', 'служебное поле pipeline, не бизнес-признак'
    if source == 'ЕГРН' and any(
        word in column.lower()
        for word in ['update', 'actual', 'status', 'registration_date']
    ):
        return 'high', 'нужно проверить, было ли значение доступно на дату договора'
    if source in {'ЕГРН', 'CDI'}:
        return 'unknown', 'нужно подтвердить исторический срез внешнего источника'
    if date_pattern.search(column):
        return 'unknown', 'нужно проверить доступность на as_of_date'
    return 'unknown', 'требуется бизнес-проверка доступности на дату расчёта'


def preliminary_decision(role, filled_pct, unique_count, quality_flags, leakage):
    if role == 'target':
        return 'target', 'целевая переменная'
    if role == 'identifier':
        return 'exclude', 'идентификатор оставляем только для связи и контроля'
    if role == 'service':
        return 'exclude', 'служебное поле не подаём в модель'
    if 'all_missing' in quality_flags:
        return 'exclude', 'колонка полностью пустая'
    if 'constant' in quality_flags:
        return 'exclude', 'в колонке одно заполненное значение'
    if leakage == 'high':
        return 'check', 'возможна утечка данных'
    if filled_pct < 5:
        return 'check', 'заполнено меньше 5% строк'
    return 'check', 'решение принимается после бизнес-проверки и baseline-модели'


# 4. Общие метрики датасета


In [ ]:
target_numeric, target_numeric_ratio = numeric_view(df[target_column])
if target_numeric_ratio < 0.95:
    raise ValueError(
        f'Колонка {target_column} не распознана как числовая'
    )

dataset_metrics = {
    'rows': len(df),
    'columns': len(df.columns),
    'target_column': target_column,
    'target_filled': int(target_numeric.notna().sum()),
    'target_missing': int(target_numeric.isna().sum()),
    'target_zero': int(target_numeric.eq(0).sum()),
    'target_positive': int(target_numeric.gt(0).sum()),
}

if 'object_id' in df.columns:
    dataset_metrics['unique_objects'] = int(df['object_id'].nunique(dropna=True))
if 'contract_id' in df.columns:
    dataset_metrics['unique_contracts'] = int(
        df['contract_id'].nunique(dropna=True)
    )
if {'task_id', 'object_id'}.issubset(df.columns):
    duplicate_mask = df.duplicated(['task_id', 'object_id'], keep=False)
    dataset_metrics['duplicate_task_object_rows'] = int(duplicate_mask.sum())
if 'full_address' in df.columns:
    dataset_metrics['full_address_filled'] = int(
        clean_string_series(df['full_address']).notna().sum()
    )
if 'cdi_is_unique_match' in df.columns:
    dataset_metrics['cdi_unique_matches'] = int(
        pd.to_numeric(df['cdi_is_unique_match'], errors='coerce').eq(1).sum()
    )
if 'egrn_is_unique_match' in df.columns:
    dataset_metrics['egrn_unique_matches'] = int(
        pd.to_numeric(df['egrn_is_unique_match'], errors='coerce').eq(1).sum()
    )
if 'cdi_match_status' in df.columns:
    dataset_metrics['cdi_ambiguous_matches'] = int(
        df['cdi_match_status'].eq('ambiguous_house_fias').sum()
    )
if 'egrn_match_method' in df.columns:
    dataset_metrics['egrn_ambiguous_matches'] = int(
        df['egrn_match_method'].eq('ambiguous').sum()
    )
if 'as_of_date' in df.columns:
    as_of = pd.to_datetime(df['as_of_date'], errors='coerce')
    dataset_metrics['as_of_date_min'] = (
        as_of.min().date().isoformat() if as_of.notna().any() else None
    )
    dataset_metrics['as_of_date_max'] = (
        as_of.max().date().isoformat() if as_of.notna().any() else None
    )

display(
    pd.DataFrame(
        dataset_metrics.items(),
        columns=['Показатель', 'Значение'],
    )
)


# 5. Паспорт всех признаков


In [ ]:
feature_rows = []

for number, feature in enumerate(df.columns, start=1):
    raw_series = df[feature]
    if pd.api.types.is_object_dtype(raw_series) or pd.api.types.is_string_dtype(raw_series):
        series = clean_string_series(raw_series)
    else:
        series = raw_series
    source = infer_source(feature)
    role = infer_role(feature)
    data_type = infer_data_type(feature, series, role)
    available_mask = source_available_mask(df, source)

    total_rows = len(series)
    source_available_rows = int(available_mask.sum())
    filled_count = int(series.notna().sum())
    missing_count = int(series.isna().sum())
    filled_pct = filled_count / total_rows * 100 if total_rows else 0.0
    filled_among_available = int(series.loc[available_mask].notna().sum())
    filled_among_available_pct = (
        filled_among_available / source_available_rows * 100
        if source_available_rows
        else np.nan
    )
    unique_count = int(series.nunique(dropna=True))
    unique_pct = unique_count / filled_count * 100 if filled_count else 0.0

    row = {
        'record_type': 'feature',
        'metric_name': None,
        'metric_value': None,
        'feature': feature,
        'russian_name': feature.replace('_', ' '),
        'source': source,
        'source_table': None,
        'source_column': None,
        'source_comment': None,
        'data_type': data_type,
        'role': role,
        'total_rows': total_rows,
        'source_available_rows': source_available_rows,
        'filled_count': filled_count,
        'missing_count': missing_count,
        'filled_pct': round(filled_pct, 4),
        'filled_among_available_pct': (
            round(filled_among_available_pct, 4)
            if not pd.isna(filled_among_available_pct)
            else np.nan
        ),
        'unique_count': unique_count,
        'unique_pct': round(unique_pct, 4),
        'zero_count': None,
        'negative_count': None,
        'outlier_iqr_count': None,
        'min': None,
        'p01': None,
        'p25': None,
        'median': None,
        'mean': None,
        'p75': None,
        'p99': None,
        'max': None,
        'top_values': None,
        'target_relation_method': None,
        'target_relation_value': None,
        'target_relation_rows': None,
        'quality_flag': None,
        'available_at_prediction_time': (
            'no' if role == 'target'
            else 'not_applicable' if role in {'identifier', 'service'}
            else 'unknown'
        ),
        'leakage_risk': None,
        'leakage_reason': None,
        'preliminary_decision': None,
        'decision_reason': None,
    }

    source_table, source_column, source_comment = dictionary_info(feature, source)
    row['source_table'] = source_table
    row['source_column'] = source_column
    row['source_comment'] = source_comment
    if source_comment is not None and not pd.isna(source_comment):
        row['russian_name'] = str(source_comment)

    quality_flags = []
    if filled_count == 0:
        quality_flags.append('all_missing')
    elif unique_count == 1:
        quality_flags.append('constant')
    if 0 < filled_pct < 5:
        quality_flags.append('coverage_lt_5pct')
    elif 5 <= filled_pct < 20:
        quality_flags.append('coverage_lt_20pct')
    if filled_count and unique_pct > 95 and role == 'feature':
        quality_flags.append('high_cardinality')

    if data_type == 'numeric':
        numeric, _ = numeric_view(series)
        valid = numeric.dropna()
        if not valid.empty:
            quantiles = valid.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
            q1 = quantiles.loc[0.25]
            q3 = quantiles.loc[0.75]
            iqr = q3 - q1
            if iqr > 0:
                outlier_count = int(
                    ((valid < q1 - 1.5 * iqr) | (valid > q3 + 1.5 * iqr)).sum()
                )
            else:
                outlier_count = 0

            row.update({
                'zero_count': int(valid.eq(0).sum()),
                'negative_count': int(valid.lt(0).sum()),
                'outlier_iqr_count': outlier_count,
                'min': valid.min(),
                'p01': quantiles.loc[0.01],
                'p25': q1,
                'median': quantiles.loc[0.50],
                'mean': valid.mean(),
                'p75': q3,
                'p99': quantiles.loc[0.99],
                'max': valid.max(),
            })
            if row['negative_count']:
                quality_flags.append('contains_negative')
            if row['zero_count']:
                quality_flags.append('contains_zero')
            if outlier_count:
                quality_flags.append('iqr_outliers')

            pair = pd.DataFrame({
                'feature': numeric,
                'target': target_numeric,
            }).dropna()
            if feature != target_column and len(pair) >= 10:
                row['target_relation_method'] = 'spearman'
                row['target_relation_value'] = pair['feature'].corr(
                    pair['target'],
                    method='spearman',
                )
                row['target_relation_rows'] = len(pair)

    elif data_type == 'datetime':
        dates, _ = datetime_view(series)
        valid_dates = dates.dropna()
        if not valid_dates.empty:
            row['min'] = valid_dates.min().isoformat()
            row['max'] = valid_dates.max().isoformat()
            pair = pd.DataFrame({
                'feature': dates.map(
                    lambda value: value.toordinal() if pd.notna(value) else np.nan
                ),
                'target': target_numeric,
            }).dropna()
            if feature != target_column and len(pair) >= 10:
                row['target_relation_method'] = 'spearman_date'
                row['target_relation_value'] = pair['feature'].corr(
                    pair['target'],
                    method='spearman',
                )
                row['target_relation_rows'] = len(pair)

    elif data_type in {'category', 'boolean'}:
        row['top_values'] = safe_top_values(
            series,
            feature,
            role,
            top_values_limit,
        )
        if role == 'feature' and 2 <= unique_count <= 50:
            eta, relation_rows = eta_squared(series, target_numeric)
            row['target_relation_method'] = 'eta_squared'
            row['target_relation_value'] = eta
            row['target_relation_rows'] = relation_rows

    elif data_type in {'text', 'json'}:
        row['top_values'] = (
            'скрыто: текстовое или конфиденциальное поле'
        )

    leakage_risk, leakage_reason = leakage_assessment(
        feature,
        source,
        role,
    )
    row['leakage_risk'] = leakage_risk
    row['leakage_reason'] = leakage_reason
    row['quality_flag'] = ' | '.join(quality_flags) if quality_flags else 'ok'

    decision, decision_reason = preliminary_decision(
        role,
        filled_pct,
        unique_count,
        quality_flags,
        leakage_risk,
    )
    row['preliminary_decision'] = decision
    row['decision_reason'] = decision_reason
    feature_rows.append(row)

    if number % 50 == 0:
        print('Обработано признаков:', number)

feature_passport = pd.DataFrame(feature_rows)
print('Признаков в паспорте:', len(feature_passport))


# 6. Добавление общих метрик в тот же CSV


In [ ]:
passport_columns = feature_passport.columns.tolist()
metric_rows = []

for metric_name, metric_value in dataset_metrics.items():
    row = {column: None for column in passport_columns}
    row['record_type'] = 'dataset_metric'
    row['metric_name'] = metric_name
    row['metric_value'] = metric_value
    metric_rows.append(row)

metric_passport = pd.DataFrame(metric_rows, columns=passport_columns)
passport = pd.concat(
    [metric_passport, feature_passport],
    ignore_index=True,
)

if len(feature_passport) != len(df.columns):
    raise ValueError('В паспорт попали не все колонки датасета')

if feature_passport['feature'].duplicated().any():
    raise ValueError('В паспорте появились повторяющиеся признаки')

display(
    feature_passport[[
        'feature',
        'source',
        'data_type',
        'role',
        'filled_pct',
        'unique_count',
        'quality_flag',
        'preliminary_decision',
    ]].head(30)
)


# 7. Контроль перед сохранением

Здесь выводятся только агрегаты. Реальные значения адресов, ИНН и идентификаторов не показываются.


In [ ]:
control = pd.DataFrame({
    'Показатель': [
        'Колонок во входном датасете',
        'Строк feature в паспорте',
        'Строк dataset_metric',
        'Всего строк итогового CSV',
        'Числовых признаков',
        'Категориальных признаков',
        'Идентификаторов',
        'Служебных полей',
        'Полностью пустых колонок',
        'Константных колонок',
        'Признаков с высоким риском утечки',
    ],
    'Значение': [
        len(df.columns),
        len(feature_passport),
        len(metric_passport),
        len(passport),
        feature_passport['data_type'].eq('numeric').sum(),
        feature_passport['data_type'].isin(['category', 'boolean']).sum(),
        feature_passport['role'].eq('identifier').sum(),
        feature_passport['role'].eq('service').sum(),
        feature_passport['quality_flag'].str.contains('all_missing').sum(),
        feature_passport['quality_flag'].str.contains('constant').sum(),
        feature_passport['leakage_risk'].eq('high').sum(),
    ],
})
display(control)


# 8. Сохранение одного CSV


In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)
passport.to_csv(
    output_file,
    index=False,
    sep=csv_separator,
    encoding=csv_encoding,
)

print('Файл сохранён:', output_file)
print('Строк:', len(passport))
print('Признаков:', len(feature_passport))
